In [3]:
#!/usr/bin/env python3
# =========================================================
# ENSEMBLE SUMMARIZATION — ROUGE-L > 0.34
# Combines ALL strategies:
#   1. BART + LED + PEGASUS + HSLN Role Classification (existing)
#   2. KG-Diff Iterative Refinement (Novel Ideas 5+7, existing)
#   3. LCS-Guided Sentence Fusion (Novel Idea 8, existing)
#   + NEW ➊: Copy-Mechanism Verbatim Span Extraction
#   + NEW ➋ (FIXED): SCST RL with BALANCED MULTI-METRIC REWARD
#                     R1 + R2 + RL  (replaces pure R1 which caused reward hacking)
#
# WHY THE FIX:
#   Previous: reward = ROUGE-1 only  → model repeated frequent unigrams
#             Result: R1=0.5087 (inflated), R2=0.2585 (low), RL=0.2456 (worst)
#             This is textbook reward hacking — high unigram overlap, broken fluency.
#
#   Fixed:    reward = α·R1 + β·R2 + γ·RL  (α=0.2, β=0.4, γ=0.4)
#             R2 penalises unigram repetition (needs bigram sequences).
#             RL penalises jumbled order (rewards sequential coherence).
#             The R2+RL dominated reward forces the model to maintain
#             both precision AND sentence-level ordering.
#
# REWARD HACKING DIAGNOSTICS:
#   A new RewardHackingMonitor class tracks per-epoch:
#     - Unigram repetition rate (should stay < 0.15)
#     - Average sentence length (should stay in [8, 40] tokens)
#     - R1/RL ratio (should stay < 2.0; high ratio = hacking signal)
#     - Vocabulary diversity (unique tokens / total tokens)
#   Training stops early if hacking is detected.
#
# EXPECTED ROUGE-L GAINS:
#   Baseline (LCS Fusion):          ~0.30–0.32
#   + Copy Mechanism ➊:             +0.02–0.04  → ~0.34+
#   + Balanced SCST RL ➋ (fixed):   +0.02–0.04  → ~0.36+  (conservative, stable)
#   Combined target:                 ROUGE-L ≥ 0.34 (conservative), ≥ 0.36 (optimistic)
# =========================================================

import os, json, re, math, copy
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from tqdm import tqdm
from nltk.tokenize import sent_tokenize, word_tokenize
from sklearn.metrics.pairwise import cosine_similarity
from transformers import (
    AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM,
    LEDForConditionalGeneration, PegasusForConditionalGeneration,
    get_linear_schedule_with_warmup
)
from torch.optim import AdamW
from collections import Counter, defaultdict
import evaluate
import nltk

# ── NLTK data ─────────────────────────────────────────────
print("📥 Downloading NLTK data...")
for resource in ['tokenizers/punkt', 'tokenizers/punkt_tab']:
    try:
        nltk.data.find(resource)
    except LookupError:
        nltk.download(resource.split('/')[-1], quiet=True)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Device: {device}")

# =========================================================
# CONFIGURATION
# =========================================================

MODEL_PATHS = {
    "BART":       "BART",
    "LED":        "LED",
    "PEGASUS":    "PEGASUS",
    "ROLE_MODEL": "best_RR_model"
}

TRAIN_JSON_PATH = "train.json"
TEST_JSON_PATH  = "test.json"
OUTPUT_DIR      = "./ensemble_results_rougeL_boost"

ROLE_CLASSIFICATION_FILE = "sentence_role_annotations_8label.json"
ROLE_CONTRIBUTION_FILE   = "role_contribution_scores_8label.json"
ROLE_WEIGHTS_FILE        = "normalized_role_weights_8label.json"

MODEL_CONFIG = {
    "BART":    {"max_input": 1024, "max_output": 256, "num_beams": 4},
    "LED":     {"max_input": 4096, "max_output": 512, "num_beams": 4},
    "PEGASUS": {"max_input": 1024, "max_output": 256, "num_beams": 4}
}

HSLN_LABELS = [
    "PREAMBLE","FAC","RLC","ISSUE","ARG_PETITIONER","ARG_RESPONDENT",
    "ANALYSIS","STA","PRE_RELIED","PRE_NOT_RELIED","RATIO","RPC","NONE"
]
NUM_HSLN_LABELS = 13

LABELS_8 = [
    "PREAMBLE","FACTS","ISSUE","ARGUMENTS",
    "REASONING","PRECEDENT_RELIED","PRECEDENT_NOT_RELIED","PROCEDURAL"
]
NUM_LABELS = 8

LABEL_MAPPING_13_TO_8 = {
    "PREAMBLE": "PREAMBLE", "ISSUE": "ISSUE", "FAC": "FACTS",
    "PRE_RELIED": "PRECEDENT_RELIED", "PRE_NOT_RELIED": "PRECEDENT_NOT_RELIED",
    "ARG_PETITIONER": "ARGUMENTS", "ARG_RESPONDENT": "ARGUMENTS",
    "ANALYSIS": "REASONING", "RATIO": "REASONING", "RPC": "REASONING",
    "STA": "PROCEDURAL", "RLC": "PROCEDURAL", "NONE": "PROCEDURAL"
}

def map_13_to_8_labels(labels_13):
    return [LABEL_MAPPING_13_TO_8.get(l, "PROCEDURAL") for l in labels_13]

HYBRID_ALPHA = 0.6
HYBRID_BETA  = 0.4

COMPRESSION_RATIOS = {"BART": 0.12, "PEGASUS": 0.12, "LED": 0.40}
MIN_SENTENCES = {"BART": 30,  "PEGASUS": 30,  "LED": 100}
MAX_SENTENCES = {"BART": 150, "PEGASUS": 150, "LED": 400}

def get_adaptive_targets(doc_length):
    targets = {}
    for model_name, ratio in COMPRESSION_RATIOS.items():
        t = int(doc_length * ratio)
        t = max(MIN_SENTENCES[model_name], t)
        t = min(MAX_SENTENCES[model_name], t)
        targets[model_name] = t
    return targets

MAX_TRAIN_SAMPLES = 3000
ENSEMBLE_WEIGHTS  = {"BART": 0.25, "LED": 0.50, "PEGASUS": 0.25}

KG_CRITICAL_ROLES     = ["ISSUE", "REASONING", "PRECEDENT_RELIED"]
KG_SEMANTIC_THRESHOLD = 0.75
KG_COVERAGE_THRESHOLD = 0.75
KG_MAX_ITERATIONS     = 2
KG_TOP_MISSING        = 5
KG_MAX_CORRECTION_SENTS = 10

LCS_ANCHOR_ROLES       = ["ISSUE", "REASONING", "PRECEDENT_RELIED", "FACTS"]
LCS_MIN_NGRAM_OVERLAP  = 3
LCS_CONNECTIVES        = [
    "The court held that", "It was observed that", "Therefore,",
    "Furthermore,", "The appellant contended that", "In view of the above,",
    "The High Court noted that", "Accordingly,"
]
LCS_MAX_OUTPUT_SENTENCES    = 20
LCS_ANCHOR_LCS_WEIGHT       = 0.5
LCS_ANCHOR_SEMANTIC_WEIGHT  = 0.5

os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================================================
# ➊  COPY-MECHANISM VERBATIM SPAN EXTRACTION  (unchanged)
# =========================================================

COPY_MIN_SPAN_TOKENS   = 4
COPY_MAX_SPAN_TOKENS   = 30
COPY_HIGH_VALUE_ROLES  = {"ISSUE", "REASONING", "PRECEDENT_RELIED"}
COPY_EXTEND_TOKENS     = 2


class SuffixArraySpanIndex:
    """
    Fast verbatim span lookup using a token-level inverted index.
    Supports finding the longest common substring between a query
    sentence and the source corpus.
    """

    def __init__(self, source_sentences, min_ngram=COPY_MIN_SPAN_TOKENS,
                 max_ngram=COPY_MAX_SPAN_TOKENS):
        self.min_ngram = min_ngram
        self.max_ngram = max_ngram
        self.source_sentences = source_sentences
        self._index: dict = defaultdict(list)
        self._tokenised_sources: list = []

        print(f"    [CopyMech] Building suffix index over {len(source_sentences)} source sentences...")
        for sent_idx, sent in enumerate(source_sentences):
            tokens = word_tokenize(sent.lower())
            self._tokenised_sources.append(tokens)
            for n in range(min_ngram, min(max_ngram + 1, len(tokens) + 1)):
                for start in range(len(tokens) - n + 1):
                    ngram_tuple = tuple(tokens[start:start + n])
                    self._index[ngram_tuple].append((sent_idx, start))

        total_entries = sum(len(v) for v in self._index.values())
        print(f"    [CopyMech] Index built: {len(self._index):,} unique n-grams, "
              f"{total_entries:,} total entries")

    def find_longest_verbatim_span(self, query_sentence):
        query_tokens = word_tokenize(query_sentence.lower())
        if len(query_tokens) < self.min_ngram:
            return {"found": False}

        best_match = {"found": False, "span_length": 0}

        for n in range(min(self.max_ngram, len(query_tokens)), self.min_ngram - 1, -1):
            for start in range(len(query_tokens) - n + 1):
                ngram_tuple = tuple(query_tokens[start:start + n])
                if ngram_tuple in self._index:
                    matches = self._index[ngram_tuple]
                    for sent_idx, src_start in matches:
                        src_tokens = self._tokenised_sources[sent_idx]
                        extended_n = n
                        ext_query_start = start
                        ext_src_start   = src_start

                        while (ext_query_start + extended_n < len(query_tokens) and
                               ext_src_start + extended_n < len(src_tokens) and
                               query_tokens[ext_query_start + extended_n] ==
                               src_tokens[ext_src_start + extended_n] and
                               extended_n < self.max_ngram):
                            extended_n += 1

                        ext_left = 0
                        while (ext_query_start - 1 >= 0 and
                               ext_src_start - 1 >= 0 and
                               query_tokens[ext_query_start - 1] ==
                               src_tokens[ext_src_start - 1] and
                               extended_n + ext_left < self.max_ngram):
                            ext_left += 1
                            ext_query_start -= 1
                            ext_src_start   -= 1

                        total_len = extended_n + ext_left

                        if total_len > best_match["span_length"]:
                            span_toks = query_tokens[ext_query_start:ext_query_start + total_len]
                            best_match = {
                                "found":           True,
                                "span_tokens":     span_toks,
                                "span_text":       " ".join(span_toks),
                                "span_length":     total_len,
                                "source_sent_idx": sent_idx,
                                "query_start":     ext_query_start,
                                "query_end":       ext_query_start + total_len
                            }

            if best_match["found"] and best_match["span_length"] >= n:
                break

        return best_match


def substitute_verbatim_span(generated_sentence, span_match, source_sentences):
    if not span_match["found"]:
        return generated_sentence

    gen_tokens = word_tokenize(generated_sentence)
    q_start    = span_match["query_start"]
    q_end      = span_match["query_end"]

    src_sent_idx     = span_match["source_sent_idx"]
    src_tokens_lower = word_tokenize(source_sentences[src_sent_idx].lower())
    src_tokens       = word_tokenize(source_sentences[src_sent_idx])
    span_lower       = span_match["span_tokens"]

    src_start = None
    for i in range(len(src_tokens_lower) - len(span_lower) + 1):
        if src_tokens_lower[i:i + len(span_lower)] == span_lower:
            src_start = i
            break

    if src_start is None:
        return generated_sentence

    verbatim_tokens = src_tokens[src_start: src_start + len(span_lower)]
    prefix  = gen_tokens[:q_start]
    suffix  = gen_tokens[q_end:]
    new_tokens = prefix + verbatim_tokens + suffix

    if new_tokens:
        new_tokens[0] = new_tokens[0].capitalize()

    return " ".join(new_tokens)


def copy_mechanism_post_process(summary, doc_annotation,
                                 span_index: SuffixArraySpanIndex,
                                 source_sentences_original: list):
    sentences = sent_tokenize(summary)
    copy_log  = {"total_sentences": len(sentences), "substitutions": 0, "details": []}

    improved_sentences = []
    for sent in sentences:
        match = span_index.find_longest_verbatim_span(sent)
        detail = {
            "original": sent[:80],
            "found":    match["found"],
            "span_len": match.get("span_length", 0)
        }
        if match["found"] and match["span_length"] >= COPY_MIN_SPAN_TOKENS:
            improved = substitute_verbatim_span(sent, match, source_sentences_original)
            detail["improved"] = improved[:80]
            detail["action"]   = "substituted"
            copy_log["substitutions"] += 1
        else:
            improved = sent
            detail["action"] = "kept"

        improved_sentences.append(improved)
        copy_log["details"].append(detail)

    improved_summary = " ".join(improved_sentences)
    return improved_summary, copy_log


def build_span_index_for_document(doc_annotation):
    high_value_sents = [
        s["sentence"] for s in doc_annotation["sentences"]
        if s["role"] in COPY_HIGH_VALUE_ROLES
    ]
    if not high_value_sents:
        high_value_sents = [s["sentence"] for s in doc_annotation["sentences"]]

    span_index = SuffixArraySpanIndex(high_value_sents)
    return span_index, high_value_sents


# =========================================================
# ➋  FIXED: SCST RL WITH BALANCED MULTI-METRIC REWARD
# ─────────────────────────────────────────────────────────
#
# ROOT CAUSE OF THE BUG:
#   The original SCST used reward = ROUGE-L only. However since ROUGE-L
#   is measured via LCS which is a global sequence metric, the gradient
#   signal was sparse and noisy at the token level. The model instead
#   latched onto the easier unigram-overlap signal (which correlates with
#   ROUGE-L at training time but diverges at test time). Result:
#     R1=0.5087 (inflated), R2=0.2585, RL=0.2456 — clear reward hacking.
#
# THE FIX — BALANCED REWARD:
#   reward = REWARD_W_R1*R1 + REWARD_W_R2*R2 + REWARD_W_RL*RL
#
#   Weights chosen with strong R2+RL bias:
#     REWARD_W_R1 = 0.20   (low — prevents unigram hacking)
#     REWARD_W_R2 = 0.40   (high — penalises repetition, needs bigrams)
#     REWARD_W_RL = 0.40   (high — rewards sequential coherence)
#
#   Why R2 prevents hacking:
#     Repeating "the court held" 5 times boosts R1 (unigrams) but
#     HURTS R2 because precision = matched_bigrams / total_bigrams:
#     repeated bigrams ("court held", "held the") likely don't appear
#     multiple times in the reference.
#
#   Why RL prevents hacking:
#     RL = LCS / len. Repeating tokens inflates length, reducing
#     precision (LCS/len_hyp). The model must be concise AND ordered.
#
# REWARD HACKING MONITOR:
#   Tracks 4 signals per epoch:
#     1. unigram_repetition_rate: repeated_tokens / total_tokens
#        Threshold: < 0.15 (healthy) / ≥ 0.15 (hacking warning)
#     2. avg_sentence_length: mean tokens per sentence
#        Healthy range: [8, 40] tokens
#     3. r1_rl_ratio: R1 / RL  (should stay < 2.0)
#        High ratio = model boosting R1 without improving RL
#     4. vocab_diversity: unique_tokens / total_tokens
#        Low diversity = repetition hacking
#
#   If 2+ signals fire, training halts early.
# =========================================================

# ── Balanced reward weights (R2 + RL dominated) ──────────
REWARD_W_R1 = 0.20    # Low weight — prevents unigram-repetition hacking
REWARD_W_R2 = 0.40    # High weight — bigram precision penalises repetition
REWARD_W_RL = 0.40    # High weight — LCS rewards sequential coherence

# ── SCST training hyperparameters ─────────────────────────
SCST_RL_WEIGHT       = 0.9995
SCST_LR              = 1e-5
SCST_EPOCHS          = 3
SCST_BATCH_SIZE      = 1
SCST_GRADIENT_ACCUM  = 8
SCST_MAX_TRAIN_SAMPS = 500
SCST_SAMPLE_TEMP     = 1.1
SCST_SAMPLE_TOP_P    = 0.9
SCST_MAX_DECODE_LEN  = 256
SCST_SAVE_PATH       = "LED_SCST"
SCST_WARMUP_STEPS    = 50

# ── Reward hacking detection thresholds ───────────────────
HACK_REPETITION_THRESHOLD  = 0.15   # unigram_repetition_rate ≥ this → warning
HACK_MIN_SENT_LEN          = 8      # avg sentence length < this → warning
HACK_MAX_SENT_LEN          = 40     # avg sentence length > this → warning
HACK_R1_RL_RATIO_MAX       = 2.0    # R1/RL ≥ this → warning
HACK_VOCAB_DIVERSITY_MIN   = 0.40   # vocab diversity < this → warning
HACK_SIGNALS_TO_STOP       = 2      # stop training if ≥ this many signals fire


# ─────────────────────────────────────────────────────────
# INLINE ROUGE FUNCTIONS
# ─────────────────────────────────────────────────────────

def _lcs_length(a_tokens: list, b_tokens: list) -> int:
    """2-row DP LCS computation."""
    m, n = len(a_tokens), len(b_tokens)
    prev = [0] * (n + 1)
    curr = [0] * (n + 1)
    for a_tok in a_tokens:
        for j, b_tok in enumerate(b_tokens):
            curr[j + 1] = prev[j] + 1 if a_tok == b_tok else max(curr[j], prev[j + 1])
        prev, curr = curr, [0] * (n + 1)
    return prev[n]


def compute_rougeL_fast(hypothesis: str, reference: str) -> float:
    """ROUGE-L F1 (token-level LCS)."""
    hyp = word_tokenize(hypothesis.lower()) if hypothesis.strip() else []
    ref = word_tokenize(reference.lower())  if reference.strip()  else []
    if not hyp or not ref:
        return 0.0
    lcs_len   = _lcs_length(hyp, ref)
    precision = lcs_len / len(hyp)
    recall    = lcs_len / len(ref)
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1


def compute_rouge1_fast(hypothesis: str, reference: str) -> float:
    """ROUGE-1 F1 (unigram overlap)."""
    hyp = Counter(word_tokenize(hypothesis.lower())) if hypothesis.strip() else Counter()
    ref = Counter(word_tokenize(reference.lower()))  if reference.strip()  else Counter()
    if not hyp or not ref:
        return 0.0
    overlap   = sum((hyp & ref).values())
    precision = overlap / sum(hyp.values())
    recall    = overlap / sum(ref.values())
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1


def compute_rouge2_fast(hypothesis: str, reference: str) -> float:
    """ROUGE-2 F1 (bigram overlap)."""
    def bigrams(tokens):
        return Counter(zip(tokens[:-1], tokens[1:]))

    hyp_toks = word_tokenize(hypothesis.lower()) if hypothesis.strip() else []
    ref_toks = word_tokenize(reference.lower())  if reference.strip()  else []
    if len(hyp_toks) < 2 or len(ref_toks) < 2:
        return 0.0
    hyp_bg = bigrams(hyp_toks)
    ref_bg = bigrams(ref_toks)
    overlap   = sum((hyp_bg & ref_bg).values())
    precision = overlap / sum(hyp_bg.values())
    recall    = overlap / sum(ref_bg.values())
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
    return f1


def compute_balanced_reward(hypothesis: str, reference: str) -> dict:
    """
    Compute the balanced multi-metric reward used in SCST training.

    Returns a dict with individual scores and the combined reward scalar.

    reward = REWARD_W_R1 * R1 + REWARD_W_R2 * R2 + REWARD_W_RL * RL

    WHY THIS PREVENTS REWARD HACKING:
      - R2 ≫ R1 weight: the model cannot inflate reward by repeating
        high-frequency unigrams because R2 will DROP (repeated bigrams
        are unlikely to match diverse reference bigrams).
      - RL ≫ R1 weight: the model must preserve sentence-level token
        order; jumbling tokens hurts LCS even if unigram overlap is high.
      - R1 is still included (small weight) to give a soft unigram signal,
        which helps training stability in early epochs.
    """
    r1 = compute_rouge1_fast(hypothesis, reference)
    r2 = compute_rouge2_fast(hypothesis, reference)
    rl = compute_rougeL_fast(hypothesis, reference)
    reward = REWARD_W_R1 * r1 + REWARD_W_R2 * r2 + REWARD_W_RL * rl
    return {"r1": r1, "r2": r2, "rl": rl, "reward": reward}


# ─────────────────────────────────────────────────────────
# REWARD HACKING MONITOR
# ─────────────────────────────────────────────────────────

class RewardHackingMonitor:
    """
    Tracks 4 per-epoch signals to detect reward hacking early.

    Signals monitored:
      1. unigram_repetition_rate:  frac. of output tokens that are repeated
         Expected healthy range:   < HACK_REPETITION_THRESHOLD (0.15)
      2. avg_sentence_length:      mean token count per generated sentence
         Expected healthy range:   [HACK_MIN_SENT_LEN, HACK_MAX_SENT_LEN]
      3. r1_rl_ratio:              R1 / RL across all samples
         Expected healthy range:   < HACK_R1_RL_RATIO_MAX (2.0)
      4. vocab_diversity:          unique_tokens / total_tokens per sample
         Expected healthy range:   ≥ HACK_VOCAB_DIVERSITY_MIN (0.40)

    If ≥ HACK_SIGNALS_TO_STOP (2) signals fire simultaneously,
    training is halted early to prevent degraded models.
    """

    def __init__(self):
        self.epoch_logs: list = []
        self._sample_buffer: list = []   # accumulate per-step (hyp, ref) pairs

    def record_sample(self, hypothesis: str, reference: str,
                       r1: float, r2: float, rl: float):
        """Call once per training step to accumulate statistics."""
        self._sample_buffer.append({
            "hyp": hypothesis,
            "ref": reference,
            "r1":  r1,
            "r2":  r2,
            "rl":  rl
        })

    def _compute_repetition_rate(self, text: str) -> float:
        tokens = word_tokenize(text.lower())
        if len(tokens) < 2:
            return 0.0
        counts    = Counter(tokens)
        repeated  = sum(v - 1 for v in counts.values() if v > 1)
        return repeated / len(tokens)

    def _compute_vocab_diversity(self, text: str) -> float:
        tokens = word_tokenize(text.lower())
        if not tokens:
            return 1.0
        return len(set(tokens)) / len(tokens)

    def _compute_avg_sent_length(self, text: str) -> float:
        sents = sent_tokenize(text)
        if not sents:
            return 0.0
        return np.mean([len(word_tokenize(s)) for s in sents])

    def compute_epoch_stats(self, epoch: int) -> dict:
        """
        Compute aggregate statistics for the current epoch and check
        for reward hacking signals. Clears the sample buffer.

        Returns a stats dict and a list of fired warning signals.
        """
        if not self._sample_buffer:
            return {}, []

        hyps = [s["hyp"] for s in self._sample_buffer]
        refs = [s["ref"] for s in self._sample_buffer]   # noqa: F841
        r1s  = [s["r1"]  for s in self._sample_buffer]
        r2s  = [s["r2"]  for s in self._sample_buffer]
        rls  = [s["rl"]  for s in self._sample_buffer]

        rep_rates  = [self._compute_repetition_rate(h)  for h in hyps]
        diversities = [self._compute_vocab_diversity(h) for h in hyps]
        sent_lens  = [self._compute_avg_sent_length(h)  for h in hyps]

        mean_r1  = float(np.mean(r1s))
        mean_r2  = float(np.mean(r2s))
        mean_rl  = float(np.mean(rls))
        mean_rep = float(np.mean(rep_rates))
        mean_div = float(np.mean(diversities))
        mean_len = float(np.mean(sent_lens))
        r1_rl_ratio = mean_r1 / max(mean_rl, 1e-6)

        stats = {
            "epoch":                  epoch,
            "mean_r1":                round(mean_r1,  4),
            "mean_r2":                round(mean_r2,  4),
            "mean_rl":                round(mean_rl,  4),
            "r1_rl_ratio":            round(r1_rl_ratio, 4),
            "unigram_repetition_rate": round(mean_rep, 4),
            "vocab_diversity":         round(mean_div, 4),
            "avg_sentence_length":     round(mean_len, 2),
            "n_samples":              len(self._sample_buffer)
        }

        # ── Detect hacking signals ────────────────────────────────────────
        warnings = []
        if mean_rep >= HACK_REPETITION_THRESHOLD:
            warnings.append(
                f"🚨 unigram_repetition_rate={mean_rep:.4f} ≥ {HACK_REPETITION_THRESHOLD} "
                f"(model repeating tokens to inflate R1)"
            )
        if not (HACK_MIN_SENT_LEN <= mean_len <= HACK_MAX_SENT_LEN):
            warnings.append(
                f"🚨 avg_sentence_length={mean_len:.1f} outside [{HACK_MIN_SENT_LEN},{HACK_MAX_SENT_LEN}] "
                f"(sentences too short/long — possible degenerate outputs)"
            )
        if r1_rl_ratio >= HACK_R1_RL_RATIO_MAX:
            warnings.append(
                f"🚨 r1_rl_ratio={r1_rl_ratio:.4f} ≥ {HACK_R1_RL_RATIO_MAX} "
                f"(R1 inflated relative to RL — unigram hacking detected)"
            )
        if mean_div < HACK_VOCAB_DIVERSITY_MIN:
            warnings.append(
                f"🚨 vocab_diversity={mean_div:.4f} < {HACK_VOCAB_DIVERSITY_MIN} "
                f"(low vocabulary diversity — repetition hacking)"
            )

        stats["warnings"]     = warnings
        stats["n_warnings"]   = len(warnings)
        stats["should_stop"]  = len(warnings) >= HACK_SIGNALS_TO_STOP

        self.epoch_logs.append(stats)
        self._sample_buffer = []   # reset for next epoch

        return stats, warnings

    def print_epoch_report(self, stats: dict, warnings: list):
        print(f"\n   📊 Reward Hacking Monitor — Epoch {stats.get('epoch', '?')}:")
        print(f"      R1={stats['mean_r1']:.4f}  R2={stats['mean_r2']:.4f}  "
              f"RL={stats['mean_rl']:.4f}  R1/RL={stats['r1_rl_ratio']:.2f}")
        print(f"      RepRate={stats['unigram_repetition_rate']:.4f}  "
              f"VocabDiv={stats['vocab_diversity']:.4f}  "
              f"AvgSentLen={stats['avg_sentence_length']:.1f}")
        if warnings:
            for w in warnings:
                print(f"      {w}")
            if stats["should_stop"]:
                print(f"      ⛔  {len(warnings)} hacking signals detected — "
                      f"early stopping training!")
        else:
            print(f"      ✅ No hacking signals — training healthy")

    def save_log(self, path: str):
        with open(path, 'w') as f:
            json.dump(self.epoch_logs, f, indent=2)
        print(f"   💾 Hacking monitor log saved: {path}")


# ─────────────────────────────────────────────────────────
# SCST CORE FUNCTIONS  (fixed reward)
# ─────────────────────────────────────────────────────────

def compute_sequence_log_probs(model, input_ids, attention_mask,
                                global_attention_mask, sampled_ids):
    """
    Compute sum of log-probabilities of sampled_ids under the model.
    Used for the policy gradient term in SCST.
    """
    decoder_input_ids = model.prepare_decoder_input_ids_from_labels(sampled_ids)

    outputs = model(
        input_ids             = input_ids,
        attention_mask        = attention_mask,
        global_attention_mask = global_attention_mask,
        decoder_input_ids     = decoder_input_ids,
        return_dict           = True
    )
    logits = outputs.logits   # [1, tgt_len, vocab_size]

    log_probs = F.log_softmax(logits, dim=-1)
    tgt_len   = sampled_ids.size(1)

    token_log_probs = log_probs[0, :tgt_len].gather(
        dim=1,
        index=sampled_ids[0, :tgt_len].unsqueeze(1)
    ).squeeze(1)

    mask = (sampled_ids[0] != model.config.pad_token_id).float()
    return (token_log_probs * mask).sum()


def scst_train_step(model, tokenizer, input_text, reference_text,
                     optimizer, hack_monitor: RewardHackingMonitor):
    """
    One SCST training step with BALANCED MULTI-METRIC REWARD.

    KEY CHANGE FROM ORIGINAL:
      reward = REWARD_W_R1*R1 + REWARD_W_R2*R2 + REWARD_W_RL*RL
               (was: reward = ROUGE-L only)

    The advantage is computed using balanced rewards for BOTH the sampled
    and greedy outputs. This ensures the policy gradient pushes the model
    towards summaries that are simultaneously:
      - High R1 (unigram overlap — but capped by R1 low weight)
      - High R2 (bigram precision — prevents repetition hacking)
      - High RL (sequential coherence — rewards ordered summaries)

    Args:
        model:          LEDForConditionalGeneration
        tokenizer:      AutoTokenizer
        input_text:     str — filtered source text
        reference_text: str — reference summary
        optimizer:      AdamW
        hack_monitor:   RewardHackingMonitor — records per-step stats

    Returns:
        total_loss:    tensor (differentiable)
        advantage:     float (balanced_reward_sample - balanced_reward_greedy)
        reward_detail: dict — r1/r2/rl breakdown for logging
    """
    model.train()
    inputs = tokenizer(
        input_text, return_tensors="pt",
        truncation=True, max_length=MODEL_CONFIG["LED"]["max_input"]
    ).to(device)

    input_ids      = inputs["input_ids"]
    attention_mask = inputs["attention_mask"]

    global_attention_mask = torch.zeros_like(attention_mask)
    global_attention_mask[:, 0] = 1

    # ── Step A: Greedy baseline (no gradient) ─────────────────────────────
    with torch.no_grad():
        greedy_ids = model.generate(
            input_ids,
            attention_mask         = attention_mask,
            global_attention_mask  = global_attention_mask,
            max_length             = SCST_MAX_DECODE_LEN,
            num_beams              = 1,
            do_sample              = False,
            no_repeat_ngram_size   = 3
        )
        greedy_text    = tokenizer.decode(greedy_ids[0], skip_special_tokens=True)
        greedy_scores  = compute_balanced_reward(greedy_text, reference_text)
        greedy_reward  = greedy_scores["reward"]

    # ── Step B: Sampled output ─────────────────────────────────────────────
    with torch.no_grad():
        sampled_ids = model.generate(
            input_ids,
            attention_mask         = attention_mask,
            global_attention_mask  = global_attention_mask,
            max_length             = SCST_MAX_DECODE_LEN,
            do_sample              = True,
            temperature            = SCST_SAMPLE_TEMP,
            top_p                  = SCST_SAMPLE_TOP_P,
            no_repeat_ngram_size   = 3
        )
        sampled_text    = tokenizer.decode(sampled_ids[0], skip_special_tokens=True)
        sampled_scores  = compute_balanced_reward(sampled_text, reference_text)
        sampled_reward  = sampled_scores["reward"]

    # ── Step C: Advantage using BALANCED reward ────────────────────────────
    # advantage > 0  → sample better than greedy  → reinforce
    # advantage < 0  → sample worse than greedy   → suppress
    advantage = sampled_reward - greedy_reward

    # ── Step D: Policy gradient loss ──────────────────────────────────────
    log_prob_sum = compute_sequence_log_probs(
        model, input_ids, attention_mask, global_attention_mask, sampled_ids
    )
    rl_loss = -advantage * log_prob_sum   # SCST loss

    # ── Step E: Cross-entropy loss (regularisation) ────────────────────────
    ref_inputs = tokenizer(
        reference_text, return_tensors="pt",
        truncation=True, max_length=SCST_MAX_DECODE_LEN
    ).to(device)
    labels = ref_inputs["input_ids"].clone()
    labels[labels == tokenizer.pad_token_id] = -100

    ce_outputs = model(
        input_ids             = input_ids,
        attention_mask        = attention_mask,
        global_attention_mask = global_attention_mask,
        labels                = labels,
        return_dict           = True
    )
    ce_loss = ce_outputs.loss

    # ── Step F: Mixed loss ─────────────────────────────────────────────────
    total_loss = SCST_RL_WEIGHT * rl_loss + (1.0 - SCST_RL_WEIGHT) * ce_loss

    # ── Record stats for hacking monitor ──────────────────────────────────
    hack_monitor.record_sample(
        hypothesis = sampled_text,
        reference  = reference_text,
        r1 = sampled_scores["r1"],
        r2 = sampled_scores["r2"],
        rl = sampled_scores["rl"]
    )

    reward_detail = {
        "greedy_r1":  greedy_scores["r1"],
        "greedy_r2":  greedy_scores["r2"],
        "greedy_rl":  greedy_scores["rl"],
        "greedy_rew": greedy_reward,
        "sample_r1":  sampled_scores["r1"],
        "sample_r2":  sampled_scores["r2"],
        "sample_rl":  sampled_scores["rl"],
        "sample_rew": sampled_reward,
        "advantage":  advantage
    }

    return total_loss, float(advantage), reward_detail


def run_scst_finetuning(led_model, led_tokenizer, train_data,
                         train_annotations, normalized_weights):
    """
    Full SCST RL fine-tuning loop with BALANCED MULTI-METRIC REWARD
    and integrated RewardHackingMonitor.

    CHANGES FROM ORIGINAL:
      1. reward = α·R1 + β·R2 + γ·RL  (replaces ROUGE-L only)
      2. RewardHackingMonitor checks 4 signals each epoch
      3. Early stopping if ≥ HACK_SIGNALS_TO_STOP signals fire
      4. Per-step R1/R2/RL logging for post-hoc analysis
      5. Saves full reward breakdown in training log JSON

    Returns:
        led_model, led_tokenizer
    """
    print("\n" + "="*70)
    print("➋  SCST RL FINE-TUNING — LED WITH BALANCED MULTI-METRIC REWARD")
    print("   (FIX: replaces ROUGE-L-only reward that caused hacking)")
    print("="*70)
    print(f"   Reward function:   {REWARD_W_R1}·R1 + {REWARD_W_R2}·R2 + {REWARD_W_RL}·RL")
    print(f"   Why not R1-only:   R2+RL bias prevents unigram repetition hacking")
    print(f"   Training samples:  {min(SCST_MAX_TRAIN_SAMPS, len(train_data))}")
    print(f"   Epochs:            {SCST_EPOCHS}")
    print(f"   RL weight:         {SCST_RL_WEIGHT}")
    print(f"   LR:                {SCST_LR}")
    print(f"   Sample temp:       {SCST_SAMPLE_TEMP}, top_p={SCST_SAMPLE_TOP_P}")
    print(f"\n   Hacking monitor thresholds:")
    print(f"     rep_rate < {HACK_REPETITION_THRESHOLD} | "
          f"sent_len ∈ [{HACK_MIN_SENT_LEN},{HACK_MAX_SENT_LEN}] | "
          f"R1/RL < {HACK_R1_RL_RATIO_MAX} | "
          f"vocab_div > {HACK_VOCAB_DIVERSITY_MIN}")
    print(f"     Stop if ≥ {HACK_SIGNALS_TO_STOP} signals fire simultaneously\n")

    # Check if already fine-tuned
    if os.path.exists(SCST_SAVE_PATH):
        print(f"   ✅ Found existing SCST model at {SCST_SAVE_PATH} — loading it")
        led_model      = LEDForConditionalGeneration.from_pretrained(SCST_SAVE_PATH).to(device)
        led_tokenizer  = AutoTokenizer.from_pretrained(SCST_SAVE_PATH)
        return led_model, led_tokenizer

    n_train = min(SCST_MAX_TRAIN_SAMPS, len(train_data), len(train_annotations))
    train_subset_data  = train_data[:n_train]
    train_subset_annot = train_annotations[:n_train]

    optimizer = AdamW(led_model.parameters(), lr=SCST_LR, weight_decay=0.01)
    total_steps = (n_train * SCST_EPOCHS) // SCST_GRADIENT_ACCUM
    scheduler   = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps   = SCST_WARMUP_STEPS,
        num_training_steps = total_steps
    )

    hack_monitor        = RewardHackingMonitor()
    epoch_history       = []
    early_stop_triggered = False

    for epoch in range(SCST_EPOCHS):
        if early_stop_triggered:
            print(f"\n⛔  Skipping epoch {epoch+1} — early stopped due to reward hacking")
            break

        print(f"\n{'─'*60}")
        print(f"  SCST Epoch {epoch+1}/{SCST_EPOCHS}")
        print(f"{'─'*60}")

        epoch_advantages   = []
        epoch_losses       = []
        epoch_reward_details = []
        optimizer.zero_grad()

        for step_idx, (train_item, doc_annotation) in enumerate(
            tqdm(zip(train_subset_data, train_subset_annot),
                 total=n_train, desc=f"Epoch {epoch+1}")
        ):
            reference_text = train_item.get("reference_summary", "")
            if not reference_text.strip():
                continue

            target_sents = get_adaptive_targets(doc_annotation["total_sentences"])["LED"]
            input_text, _ = hybrid_role_salience_selection(
                doc_annotation, normalized_weights, target_sents
            )
            if not input_text.strip():
                continue

            try:
                step_loss, advantage, reward_detail = scst_train_step(
                    led_model, led_tokenizer,
                    input_text, reference_text,
                    optimizer, hack_monitor
                )

                step_loss = step_loss / SCST_GRADIENT_ACCUM
                step_loss.backward()

                epoch_advantages.append(advantage)
                epoch_losses.append(step_loss.item() * SCST_GRADIENT_ACCUM)
                epoch_reward_details.append(reward_detail)

                if (step_idx + 1) % SCST_GRADIENT_ACCUM == 0:
                    torch.nn.utils.clip_grad_norm_(led_model.parameters(), max_norm=1.0)
                    optimizer.step()
                    scheduler.step()
                    optimizer.zero_grad()

            except torch.cuda.OutOfMemoryError:
                print(f"\n    ⚠️  OOM at step {step_idx}, skipping...")
                torch.cuda.empty_cache()
                optimizer.zero_grad()
                continue
            except Exception as e:
                print(f"\n    ⚠️  Error at step {step_idx}: {e}")
                continue

        # Final gradient step
        if n_train % SCST_GRADIENT_ACCUM != 0:
            torch.nn.utils.clip_grad_norm_(led_model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad()

        # ── Epoch summary ──────────────────────────────────────────────────
        mean_adv  = float(np.mean(epoch_advantages)) if epoch_advantages else 0.0
        mean_loss = float(np.mean(epoch_losses))      if epoch_losses     else 0.0
        pos_pct   = sum(1 for a in epoch_advantages if a > 0) / max(len(epoch_advantages), 1) * 100

        if epoch_reward_details:
            mean_r1  = float(np.mean([d["sample_r1"]  for d in epoch_reward_details]))
            mean_r2  = float(np.mean([d["sample_r2"]  for d in epoch_reward_details]))
            mean_rl  = float(np.mean([d["sample_rl"]  for d in epoch_reward_details]))
        else:
            mean_r1 = mean_r2 = mean_rl = 0.0

        print(f"\n   Epoch {epoch+1} results:")
        print(f"     Mean advantage (Δbalanced_reward): {mean_adv:+.4f}")
        print(f"     Mean sample R1={mean_r1:.4f}  R2={mean_r2:.4f}  RL={mean_rl:.4f}")
        print(f"     Mean total loss:                   {mean_loss:.4f}")
        print(f"     Sample > greedy:                   {pos_pct:.1f}%")

        # ── Reward hacking monitor ─────────────────────────────────────────
        stats, warnings = hack_monitor.compute_epoch_stats(epoch + 1)
        hack_monitor.print_epoch_report(stats, warnings)

        epoch_history.append({
            "epoch":          epoch + 1,
            "mean_advantage": mean_adv,
            "mean_loss":      mean_loss,
            "mean_r1":        mean_r1,
            "mean_r2":        mean_r2,
            "mean_rl":        mean_rl,
            "hack_stats":     stats
        })

        if stats.get("should_stop", False):
            print(f"\n⛔  EARLY STOPPING: reward hacking detected at epoch {epoch+1}")
            early_stop_triggered = True

    # ── Save fine-tuned model ──────────────────────────────────────────────
    print(f"\n💾 Saving SCST fine-tuned LED to: {SCST_SAVE_PATH}")
    os.makedirs(SCST_SAVE_PATH, exist_ok=True)
    led_model.save_pretrained(SCST_SAVE_PATH)
    led_tokenizer.save_pretrained(SCST_SAVE_PATH)

    training_log = {
        "scst_config": {
            "reward_weights":   {"r1": REWARD_W_R1, "r2": REWARD_W_R2, "rl": REWARD_W_RL},
            "rl_weight":        SCST_RL_WEIGHT,
            "lr":               SCST_LR,
            "epochs":           SCST_EPOCHS,
            "train_samples":    n_train,
            "sample_temp":      SCST_SAMPLE_TEMP,
            "sample_top_p":     SCST_SAMPLE_TOP_P,
            "early_stopped":    early_stop_triggered,
            "fix_description":  (
                "Replaced ROUGE-L-only reward (which caused R1 inflation) with "
                f"balanced reward: {REWARD_W_R1}*R1 + {REWARD_W_R2}*R2 + {REWARD_W_RL}*RL. "
                "R2+RL dominance prevents unigram repetition hacking."
            )
        },
        "epoch_history": epoch_history,
        "hack_monitor_log": hack_monitor.epoch_logs
    }

    log_path = os.path.join(SCST_SAVE_PATH, "scst_training_log.json")
    with open(log_path, 'w') as f:
        json.dump(training_log, f, indent=2)

    hack_monitor.save_log(os.path.join(OUTPUT_DIR, "reward_hacking_monitor_log.json"))

    print(f"✅ SCST fine-tuning complete (early_stopped={early_stop_triggered}). Model saved.")
    return led_model, led_tokenizer


# =========================================================
# HSLN MODEL DEFINITION  (unchanged)
# =========================================================

class PrototypeAttention(nn.Module):
    def __init__(self, dim, heads=8, dropout=0.1):
        super().__init__()
        self.h  = heads
        self.dh = dim // heads
        self.q  = nn.Linear(dim, dim, bias=False)
        self.k  = nn.Linear(dim, dim, bias=False)
        self.v  = nn.Linear(dim, dim, bias=False)
        self.o  = nn.Linear(dim, dim, bias=False)
        self.ln = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, p):
        B, T, D = x.shape; C = p.size(0)
        Q = self.q(x).view(B, T, self.h, self.dh)
        K = self.k(p).view(C, self.h, self.dh)
        V = self.v(p).view(C, self.h, self.dh)
        Q = F.normalize(Q, dim=-1); K = F.normalize(K, dim=-1)
        outs = []
        for h in range(self.h):
            a = (Q[:, :, h] @ K[:, h].T).softmax(-1)
            a = self.dropout(a)
            outs.append(a @ V[:, h])
        out = torch.cat(outs, -1)
        return self.ln(x + self.dropout(self.o(out)))

class RPL(nn.Module):
    def __init__(self, dim, num_labels):
        super().__init__()
        self.proto = nn.Parameter(torch.randn(num_labels, dim))
    def forward(self, h):
        return F.normalize(h, dim=-1) @ F.normalize(self.proto, dim=-1).T

class RTM(nn.Module):
    def __init__(self, num_labels, rtm_lambda):
        super().__init__()
        self.A = nn.Parameter(torch.zeros(num_labels, num_labels))
        self.rtm_lambda = rtm_lambda
    def forward(self, logits):
        lp = logits.log_softmax(-1)
        for t in range(1, logits.size(1)):
            tr = torch.logsumexp(lp[:, t-1].unsqueeze(1) + self.A.log_softmax(-1), -1)
            logits[:, t] += self.rtm_lambda * tr
        return logits

class HSLNModel(nn.Module):
    def __init__(self, embedding_dim=768, lstm_hidden=384, num_labels=13,
                 dropout=0.3, rtm_lambda=0.05):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.lstm_hidden   = lstm_hidden
        self.num_labels    = num_labels
        self.att  = PrototypeAttention(embedding_dim, dropout=dropout)
        self.lstm = nn.LSTM(embedding_dim, lstm_hidden, 2,
                            bidirectional=True, batch_first=True, dropout=dropout)
        self.cls  = nn.Linear(lstm_hidden * 2, num_labels)
        self.rpl  = RPL(lstm_hidden * 2, num_labels)
        self.alpha = nn.Parameter(torch.tensor(2.0))
        self.rtm  = RTM(num_labels, rtm_lambda)
        self.register_buffer('prototypes', torch.randn(num_labels, embedding_dim))

    def forward(self, embeddings, lengths=None):
        x = self.att(embeddings, self.prototypes)
        if lengths is not None:
            pck = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
            o, _ = self.lstm(pck)
            o, _ = nn.utils.rnn.pad_packed_sequence(o, batch_first=True)
        else:
            o, _ = self.lstm(x)
        l1 = self.cls(o); l2 = self.rpl(o)
        a  = torch.sigmoid(self.alpha)
        return self.rtm(a * l1 + (1 - a) * l2)

    def predict(self, embeddings, lengths=None):
        return torch.argmax(self.forward(embeddings, lengths), dim=-1)


# =========================================================
# LOAD MODELS
# =========================================================

print("\n📚 Loading HSLN role classifier...")
config_path = os.path.join(MODEL_PATHS["ROLE_MODEL"], "config.json")
if os.path.exists(config_path):
    with open(config_path) as f:
        cfg = json.load(f)
    embedding_dim = cfg.get('embedding_dim', 768)
    lstm_hidden   = cfg.get('lstm_hidden', 384)
    dropout       = cfg.get('dropout', 0.3)
    rtm_lambda    = cfg.get('rtm_lambda', 0.05)
else:
    embedding_dim, lstm_hidden, dropout, rtm_lambda = 768, 384, 0.3, 0.05

role_model = HSLNModel(embedding_dim, lstm_hidden, NUM_HSLN_LABELS, dropout, rtm_lambda)
state_dict = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"], "pytorch_model.bin"), map_location=device)
if 'prototypes' in state_dict: del state_dict['prototypes']
role_model.load_state_dict(state_dict, strict=False)
role_model.prototypes = torch.load(os.path.join(MODEL_PATHS["ROLE_MODEL"], "prototypes.pt"), map_location=device)
role_model.to(device).eval()
print("✅ HSLN loaded")

print("\n📚 Loading InLegalBERT...")
legal_tokenizer = AutoTokenizer.from_pretrained("law-ai/InLegalBERT")
legal_model     = AutoModel.from_pretrained("law-ai/InLegalBERT").to(device).eval()
print("✅ InLegalBERT loaded")

print("\n📚 Loading summarization models...")
models     = {}
tokenizers = {}
for mname, mclass in [("BART", AutoModelForSeq2SeqLM),
                       ("LED",  LEDForConditionalGeneration),
                       ("PEGASUS", PegasusForConditionalGeneration)]:
    print(f"  Loading {mname}...")
    tokenizers[mname] = AutoTokenizer.from_pretrained(MODEL_PATHS[mname])
    models[mname]     = mclass.from_pretrained(MODEL_PATHS[mname]).to(device).eval()
    print(f"  ✅ {mname} loaded")
print("✅ All models loaded")

print("\n📊 Loading evaluation metrics...")
rouge_metric     = evaluate.load("rouge")
bertscore_metric = evaluate.load("bertscore")


# =========================================================
# EMBEDDING & ROLE CLASSIFICATION  (unchanged)
# =========================================================

@torch.no_grad()
def embed_with_legalbert(texts, batch_size=16):
    if not texts: return np.array([])
    embs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        enc   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=512, return_tensors="pt").to(device)
        out   = legal_model(**enc).last_hidden_state
        mask  = enc["attention_mask"].unsqueeze(-1).float()
        embs.append(((out * mask).sum(1) / mask.sum(1)).cpu().numpy())
    return np.vstack(embs)

@torch.no_grad()
def classify_roles(sentences, batch_size=16):
    if not sentences: return []
    all_preds = []
    for i in range(0, len(sentences), batch_size):
        batch = sentences[i:i+batch_size]
        inp   = legal_tokenizer(batch, padding=True, truncation=True,
                                max_length=128, return_tensors="pt").to(device)
        emb   = legal_model(**inp).last_hidden_state.mean(dim=1).unsqueeze(1)
        lens  = torch.ones(len(batch), dtype=torch.long).to(device)
        preds = role_model.predict(emb, lens)[:, 0].cpu().tolist()
        all_preds.extend(preds)
    return map_13_to_8_labels([HSLN_LABELS[p] for p in all_preds])


# =========================================================
# HYBRID SELECTION  (unchanged)
# =========================================================

def hybrid_role_salience_selection(doc_annotation, normalized_weights, target_sentences):
    sentences = [s["sentence"] for s in doc_annotation["sentences"]]
    roles     = [s["role"]     for s in doc_annotation["sentences"]]
    if not sentences:
        return "", {}
    embeddings   = embed_with_legalbert(sentences)
    centroid     = embeddings.mean(axis=0, keepdims=True)
    similarities = cosine_similarity(embeddings, centroid).squeeze()
    scored = []
    for idx, (role, sim) in enumerate(zip(roles, similarities)):
        scored.append({
            "index": idx, "sentence": sentences[idx], "role": role,
            "score": HYBRID_ALPHA * normalized_weights.get(role, 0) * 10 + HYBRID_BETA * float(sim)
        })
    selected = sorted(sorted(scored, key=lambda x: x["score"], reverse=True)[:target_sentences],
                      key=lambda x: x["index"])
    return " ".join(s["sentence"] for s in selected), {"selected": len(selected)}


# =========================================================
# KG FUNCTIONS  (unchanged)
# =========================================================

def extract_triples_as_tuples(sentences):
    triples = []
    try:
        import spacy
        try: nlp = spacy.load("en_legal_ner_trf")
        except OSError:
            try: nlp = spacy.load("en_core_web_sm")
            except OSError: return _extract_triples_fallback(sentences)
        for sent in sentences:
            if not sent.strip(): continue
            doc = nlp(sent[:512])
            for token in doc:
                if token.dep_ == "ROOT" and token.pos_ == "VERB":
                    subjs = [c for c in token.children if c.dep_ in ("nsubj","nsubjpass")]
                    objs  = [c for c in token.children if c.dep_ in ("dobj","pobj","attr","oprd")]
                    for s in subjs:
                        for o in objs:
                            st = " ".join(t.text for t in s.subtree if not t.is_stop).lower().strip()
                            ot = " ".join(t.text for t in o.subtree  if not t.is_stop).lower().strip()
                            rt = token.lemma_.lower()
                            if st and ot and rt: triples.append((st, rt, ot))
    except ImportError:
        triples = _extract_triples_fallback(sentences)
    return triples

def _extract_triples_fallback(sentences):
    triples = []
    for sent in sentences:
        words = [w.lower() for w in re.findall(r'\b[A-Za-z][a-z]+(?:\s+[A-Z][a-z]+)*\b', sent) if len(w) > 3]
        for i in range(len(words)-1): triples.append((words[i],"related_to",words[i+1]))
    return triples

def triple_to_text(t): return f"{t[0]} {t[1]} {t[2]}"

def compute_semantic_kg_coverage(critical_triples, summary_triples, threshold=KG_SEMANTIC_THRESHOLD):
    if not critical_triples: return 1.0, [], []
    if not summary_triples:  return 0.0, [(t,0.0,False) for t in critical_triples], [(t,0.0) for t in critical_triples]
    ct  = [triple_to_text(t) for t in critical_triples]
    st  = [triple_to_text(t) for t in summary_triples]
    sim = cosine_similarity(embed_with_legalbert(ct), embed_with_legalbert(st))
    per, uncovered, covered = [], [], 0
    for i, ct_triple in enumerate(critical_triples):
        mx = float(sim[i].max()); cov = mx >= threshold
        per.append((ct_triple, mx, cov))
        if cov: covered += 1
        else:   uncovered.append((ct_triple, mx))
    uncovered.sort(key=lambda x: x[1])
    return covered / len(critical_triples), per, uncovered

def get_missing_entities_from_uncovered(uncovered_triples, top_k=KG_TOP_MISSING):
    ents = set()
    for (subj,rel,obj), _ in uncovered_triples[:top_k]:
        for tok in subj.split() + obj.split():
            if len(tok) > 3: ents.add(tok.lower())
    return ents

def kg_iterative_refinement(summaries_dict, doc_annotation, max_iterations=KG_MAX_ITERATIONS):
    critical_sents = [s["sentence"] for s in doc_annotation["sentences"] if s["role"] in KG_CRITICAL_ROLES]
    if not critical_sents: return summaries_dict.get("LED",""), {}
    critical_triples = extract_triples_as_tuples(critical_sents)
    if not critical_triples: return summaries_dict.get("LED",""), {}
    current_summary = summaries_dict.get("LED","") or summaries_dict.get("PEGASUS","")
    for _ in range(max_iterations):
        coverage, _, uncovered = compute_semantic_kg_coverage(critical_triples, extract_triples_as_tuples(sent_tokenize(current_summary)))
        if coverage >= KG_COVERAGE_THRESHOLD or not uncovered: break
        missing = get_missing_entities_from_uncovered(uncovered)
        correction_sents = [s["sentence"] for s in doc_annotation["sentences"]
                            if any(e in s["sentence"].lower() for e in missing)][:KG_MAX_CORRECTION_SENTS]
        if not correction_sents: break
        refined = generate_summary("PEGASUS",
            f"Improve this legal summary by including the missing information.\n\n"
            f"Current summary: {current_summary}\n\n"
            f"Missing information: {' '.join(correction_sents)}\n\nImproved summary:")
        new_coverage, _, _ = compute_semantic_kg_coverage(critical_triples, extract_triples_as_tuples(sent_tokenize(refined)))
        if new_coverage > coverage: current_summary = refined
    return current_summary, {}


# =========================================================
# LCS-GUIDED SENTENCE FUSION  (unchanged)
# =========================================================

def compute_token_lcs_length(a, b):
    if not a or not b: return 0
    if len(a) < len(b): a, b = b, a
    n = len(b); prev = [0]*(n+1); curr = [0]*(n+1)
    for ta in a:
        for j, tb in enumerate(b):
            curr[j+1] = prev[j]+1 if ta.lower()==tb.lower() else max(curr[j], prev[j+1])
        prev, curr = curr, [0]*(n+1)
    return prev[n]

def compute_lcs_score(sentence, source_sentences):
    if not source_sentences: return 0.0
    st = word_tokenize(sentence.lower())
    if not st: return 0.0
    best = 0.0
    for src in source_sentences:
        sr = word_tokenize(src.lower())
        if not sr: continue
        lcs = compute_token_lcs_length(st, sr)
        best = max(best, lcs / max(len(st), len(sr)))
    return best

def find_source_position(sentence, doc_annotation):
    best_pos, best_score = 999, -1.0
    st = word_tokenize(sentence.lower())
    for s in doc_annotation["sentences"]:
        lcs = compute_token_lcs_length(st, word_tokenize(s["sentence"].lower()))
        if lcs > best_score: best_score = lcs; best_pos = s["index"]
    return best_pos

def fuse_adjacent_sentences(sa, sb, min_ngram=LCS_MIN_NGRAM_OVERLAP):
    ta = word_tokenize(sa.lower()); tb = word_tokenize(sb.lower())
    max_check = min(15, len(ta), len(tb))
    for n in range(max_check, min_ngram-1, -1):
        if ta[-n:] == tb[:n]:
            orig_b = word_tokenize(sb)
            tail   = orig_b[n:]
            if tail:
                tail[0] = tail[0].lower()
                return f"{sa.rstrip('. ')}, {' '.join(tail)}"
    return f"{sa} {sb}"

def inject_connectives(sentences, connectives=LCS_CONNECTIVES):
    if not sentences: return sentences
    triggers = {"it","this","they","the same","such","these","those"}
    result = [sentences[0]]; ci = 0
    for sent in sentences[1:]:
        fw = sent.split()[0].lower() if sent.split() else ""
        if fw in triggers:
            conn = connectives[ci % len(connectives)]; ci += 1
            result.append(f"{conn} {sent[0].lower()}{sent[1:]}" if conn.endswith("that") else f"{conn} {sent}")
        else:
            result.append(sent)
    return result

def lcs_guided_sentence_fusion(kg_refined_summary, doc_annotation, normalized_weights):
    anchor_source_sents = [s["sentence"] for s in doc_annotation["sentences"] if s["role"] in LCS_ANCHOR_ROLES]
    if not anchor_source_sents:
        anchor_source_sents = [s["sentence"] for s in doc_annotation["sentences"]]
    summary_sentences = sent_tokenize(kg_refined_summary)
    if not summary_sentences: return kg_refined_summary, {}
    sum_emb = embed_with_legalbert(summary_sentences)
    src_emb = embed_with_legalbert(anchor_source_sents)
    sem_scores = cosine_similarity(sum_emb, src_emb).max(axis=1)
    scored = [{"sentence": s,
               "anchor_score": LCS_ANCHOR_LCS_WEIGHT * compute_lcs_score(s, anchor_source_sents)
                               + LCS_ANCHOR_SEMANTIC_WEIGHT * float(sem_scores[i])}
              for i, s in enumerate(summary_sentences)]
    selected = sorted(scored, key=lambda x: x["anchor_score"], reverse=True)[:LCS_MAX_OUTPUT_SENTENCES]
    for item in selected:
        item["source_pos"] = find_source_position(item["sentence"], doc_annotation)
    ordered = [item["sentence"] for item in sorted(selected, key=lambda x: x["source_pos"])]
    if len(ordered) >= 2:
        fused = [ordered[0]]
        for i in range(1, len(ordered)):
            merged = fuse_adjacent_sentences(fused[-1], ordered[i])
            if merged != f"{fused[-1]} {ordered[i]}": fused[-1] = merged
            else: fused.append(ordered[i])
    else:
        fused = ordered
    final = inject_connectives(fused)
    return " ".join(final), {"sentences": len(final)}


# =========================================================
# SUMMARY GENERATION
# =========================================================

def generate_summary(model_name, filtered_text, use_scst_led=False):
    if use_scst_led and model_name == "LED":
        model     = models["LED_SCST"]
        tokenizer = tokenizers["LED_SCST"]
    else:
        model     = models[model_name]
        tokenizer = tokenizers[model_name]
    config = MODEL_CONFIG[model_name]
    inputs = tokenizer(filtered_text, return_tensors="pt",
                       truncation=True, max_length=config["max_input"]).to(device)
    with torch.no_grad():
        if model_name == "LED":
            gam = torch.zeros_like(inputs["attention_mask"])
            gam[:, 0] = 1
            ids = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                  global_attention_mask=gam, max_length=config["max_output"],
                                  num_beams=config["num_beams"], early_stopping=True, no_repeat_ngram_size=3)
        else:
            ids = model.generate(inputs["input_ids"], attention_mask=inputs["attention_mask"],
                                  max_length=config["max_output"], num_beams=config["num_beams"],
                                  early_stopping=True, no_repeat_ngram_size=3)
    return tokenizer.decode(ids[0], skip_special_tokens=True)


# =========================================================
# ENSEMBLE STRATEGIES  (unchanged)
# =========================================================

def ensemble_weighted_concat(summaries_dict, weights):
    parts = []
    for mn, summary in summaries_dict.items():
        sents = sent_tokenize(summary); n = max(1, int(len(sents)*weights[mn]))
        parts.extend(sents[:n])
    seen = set(); unique = []
    for s in parts:
        norm = s.lower().strip()
        if norm not in seen: seen.add(norm); unique.append(s)
    return " ".join(unique)

def ensemble_voting(summaries_dict):
    all_sents = []
    for s in summaries_dict.values(): all_sents.extend(sent_tokenize(s))
    counts = Counter(s.lower().strip() for s in all_sents)
    selected = [s for s, c in counts.items() if c >= 2]
    if len(selected) < 3: selected = [s for s, _ in counts.most_common(10)]
    return " ".join(selected)

def ensemble_sentence_ranking(summaries_dict):
    positions = {}
    for _, summary in summaries_dict.items():
        for pos, sent in enumerate(sent_tokenize(summary)):
            norm = sent.lower().strip()
            positions.setdefault(norm, []).append(pos)
    ranked = sorted(positions.items(), key=lambda x: np.mean(x[1]))
    return " ".join(s for s, _ in ranked[:15])


# =========================================================
# EVALUATION
# =========================================================

def compute_metrics(predictions, references):
    rouge = rouge_metric.compute(predictions=predictions, references=references, use_stemmer=True)
    bert  = bertscore_metric.compute(predictions=predictions, references=references,
                                      model_type="roberta-large", lang="en", device=device, batch_size=8)
    return {
        "rouge1": float(rouge["rouge1"]), "rouge2": float(rouge["rouge2"]),
        "rougeL": float(rouge["rougeL"]), "bertscore_f1": float(np.mean(bert["f1"]))
    }


# =========================================================
# ROLE ANNOTATION + CONTRIBUTION SCORING  (unchanged)
# =========================================================

def create_role_annotations(data, output_file):
    print(f"\n📝 Annotating {len(data)} documents with 8-label roles...")
    annotations = []
    for idx, item in enumerate(tqdm(data)):
        judgment  = item.get("judgment", "")
        sentences = sent_tokenize(judgment)
        if not sentences: continue
        roles     = classify_roles(sentences)
        annotations.append({
            "doc_id": item.get("id", idx),
            "total_sentences": len(sentences),
            "sentences": [{"index": i, "sentence": s, "role": r}
                          for i, (s, r) in enumerate(zip(sentences, roles))]
        })
    with open(output_file, 'w', encoding='utf8') as f:
        json.dump(annotations, f, indent=2, ensure_ascii=False)
    print(f"✅ Annotations saved: {output_file} ({len(annotations)} docs)")
    return annotations

def calculate_role_contribution(train_annotations, train_data, output_file):
    print(f"\n📊 Calculating role contribution scores...")
    role_total = Counter(); role_sum = Counter()
    for ann, item in tqdm(zip(train_annotations, train_data), total=len(train_annotations)):
        ref_sents = sent_tokenize(item.get("reference_summary", ""))
        doc_sents = [s["sentence"] for s in ann["sentences"]]
        doc_roles = [s["role"]     for s in ann["sentences"]]
        for r in doc_roles: role_total[r] += 1
        if doc_sents and ref_sents:
            doc_emb = embed_with_legalbert(doc_sents)
            ref_emb = embed_with_legalbert(ref_sents)
            for re in ref_emb:
                sims = cosine_similarity([re], doc_emb)[0]
                best = np.argmax(sims)
                if sims[best] > 0.7: role_sum[doc_roles[best]] += 1
    role_scores = {r: role_sum[r]/role_total[r] if role_total[r] > 0 else 0.0 for r in LABELS_8}
    data = {"label_system":"8 labels","role_scores":role_scores,
            "role_total_counts":dict(role_total),"role_summary_counts":dict(role_sum)}
    with open(output_file,'w',encoding='utf8') as f: json.dump(data, f, indent=2)
    print(f"✅ Contribution scores saved: {output_file}")
    return role_scores

def normalize_role_weights(role_scores, output_file):
    total = sum(role_scores.values())
    nw    = {r: s/total if total > 0 else 1/len(LABELS_8) for r, s in role_scores.items()}
    with open(output_file,'w',encoding='utf8') as f:
        json.dump({"label_system":"8 labels","normalized_weights":nw,"original_scores":role_scores}, f, indent=2)
    print(f"✅ Normalized weights saved: {output_file}")
    return nw


# =========================================================
# MAIN PIPELINE
# =========================================================

def main():
    print("\n" + "="*70)
    print("🚀 ENSEMBLE SUMMARIZATION — ROUGE-L > 0.34")
    print("   + ➊ Copy-Mechanism Verbatim Span Extraction")
    print("   + ➋ SCST RL (FIXED: balanced R1+R2+RL reward)")
    print("      FIX: replaces ROUGE-L-only reward that caused hacking")
    print("           (was: R1=0.51 inflated, RL=0.24 worst)")
    print("           (fix: 0.2·R1 + 0.4·R2 + 0.4·RL)")
    print("   + Novel Idea 8: LCS-Guided Sentence Fusion (existing)")
    print("="*70)

    # ── Load data ──────────────────────────────────────────────────────────
    print(f"\n📂 Loading data...")
    with open(TRAIN_JSON_PATH, encoding='utf8') as f: train_data_full = json.load(f)
    with open(TEST_JSON_PATH,  encoding='utf8') as f: test_data       = json.load(f)

    # SCST fine-tuning uses a capped subset (memory / time budget)
    train_data_scst = train_data_full[:MAX_TRAIN_SAMPLES]

    print(f"   Full train set : {len(train_data_full)} documents  (used for role contribution)")
    print(f"   SCST train set : {len(train_data_scst)} documents  (capped by MAX_TRAIN_SAMPLES)")
    print(f"   Test set       : {len(test_data)} documents")

    # ── Role annotations ───────────────────────────────────────────────────
    # Annotations are computed over the FULL training set so that role
    # contribution scores reflect the entire data distribution, not just
    # the first MAX_TRAIN_SAMPLES documents.
    train_annot_path      = os.path.join(OUTPUT_DIR, ROLE_CLASSIFICATION_FILE)
    train_annot_scst_path = os.path.join(OUTPUT_DIR, "scst_" + ROLE_CLASSIFICATION_FILE)
    test_annot_path       = os.path.join(OUTPUT_DIR, "test_" + ROLE_CLASSIFICATION_FILE)
    contrib_path          = os.path.join(OUTPUT_DIR, ROLE_CONTRIBUTION_FILE)
    weights_path          = os.path.join(OUTPUT_DIR, ROLE_WEIGHTS_FILE)

    # Full-set annotations (for role contribution calculation)
    if not os.path.exists(train_annot_path):
        print(f"\n📝 Annotating FULL training set ({len(train_data_full)} docs)...")
        create_role_annotations(train_data_full, train_annot_path)

    # SCST-subset annotations (needed by run_scst_finetuning)
    if not os.path.exists(train_annot_scst_path):
        print(f"\n📝 Annotating SCST training subset ({len(train_data_scst)} docs)...")
        create_role_annotations(train_data_scst, train_annot_scst_path)

    # Test annotations
    if not os.path.exists(test_annot_path):
        create_role_annotations(test_data, test_annot_path)

    with open(train_annot_path)      as f: train_annotations_full = json.load(f)
    with open(train_annot_scst_path) as f: train_annotations_scst = json.load(f)
    with open(test_annot_path)       as f: test_annotations       = json.load(f)

    # Role contribution uses the FULL training set annotations + data
    if os.path.exists(contrib_path):
        with open(contrib_path) as f: role_scores = json.load(f)["role_scores"]
        print(f"\n✅ Loaded existing role contribution scores from {contrib_path}")
    else:
        print(f"\n📊 Calculating role contribution on FULL training set "
              f"({len(train_data_full)} docs)...")
        role_scores = calculate_role_contribution(
            train_annotations_full, train_data_full, contrib_path
        )

    if os.path.exists(weights_path):
        with open(weights_path) as f: normalized_weights = json.load(f)["normalized_weights"]
        print(f"✅ Loaded existing normalized weights from {weights_path}")
    else:
        normalized_weights = normalize_role_weights(role_scores, weights_path)

    # ── ➋ FIXED SCST RL Fine-tuning ───────────────────────────────────────
    print("\n" + "="*70)
    print("➋ RUNNING FIXED SCST RL FINE-TUNING (balanced reward)")
    print("   Reward: 0.20·R1 + 0.40·R2 + 0.40·RL")
    print(f"   Training subset: {len(train_data_scst)} / {len(train_data_full)} docs")
    print("="*70)
    models["LED"], tokenizers["LED"] = run_scst_finetuning(
        models["LED"], tokenizers["LED"],
        train_data_scst, train_annotations_scst, normalized_weights
    )
    models["LED_SCST"]     = models["LED"]
    tokenizers["LED_SCST"] = tokenizers["LED"]

    # ── Generate summaries ─────────────────────────────────────────────────
    print("\n" + "="*70)
    print("📝 GENERATING SUMMARIES")
    print("="*70)

    all_model_summaries = {k: [] for k in ["BART","LED","PEGASUS","LED_SCST"]}
    ensemble_summaries  = {k: [] for k in ["voting","weighted","ranking",
                                            "kg_refined","lcs_fused",
                                            "copy_mech",
                                            "scst_led_copy",
                                            "full_pipeline"]}
    references   = []
    copy_logs    = []
    fusion_logs  = []

    for test_annotation, test_item in tqdm(zip(test_annotations, test_data),
                                            total=len(test_data), desc="Processing"):
        reference = test_item["reference_summary"]
        references.append(reference)

        doc_length       = test_annotation["total_sentences"]
        adaptive_targets = get_adaptive_targets(doc_length)

        span_index, source_sents_orig = build_span_index_for_document(test_annotation)

        summaries_dict = {}
        for model_name in ["BART","LED","PEGASUS"]:
            target        = adaptive_targets[model_name]
            filtered_text, _ = hybrid_role_salience_selection(
                test_annotation, normalized_weights, target)
            summary = generate_summary(model_name, filtered_text, use_scst_led=False)
            all_model_summaries[model_name].append(summary)
            summaries_dict[model_name] = summary

        target_led    = adaptive_targets["LED"]
        led_input, _  = hybrid_role_salience_selection(test_annotation, normalized_weights, target_led)
        scst_summary  = generate_summary("LED", led_input, use_scst_led=True)
        all_model_summaries["LED_SCST"].append(scst_summary)

        ensemble_summaries["voting"].append(ensemble_voting(summaries_dict))
        ensemble_summaries["weighted"].append(ensemble_weighted_concat(summaries_dict, ENSEMBLE_WEIGHTS))
        ensemble_summaries["ranking"].append(ensemble_sentence_ranking(summaries_dict))

        kg_refined, _ = kg_iterative_refinement(summaries_dict, test_annotation)
        ensemble_summaries["kg_refined"].append(kg_refined)

        lcs_fused, flog = lcs_guided_sentence_fusion(kg_refined, test_annotation, normalized_weights)
        ensemble_summaries["lcs_fused"].append(lcs_fused)
        if len(fusion_logs) < 5: fusion_logs.append({"doc_id": test_annotation["doc_id"], "log": flog})

        copy_improved, clog = copy_mechanism_post_process(
            lcs_fused, test_annotation, span_index, source_sents_orig)
        ensemble_summaries["copy_mech"].append(copy_improved)
        if len(copy_logs) < 5: copy_logs.append({"doc_id": test_annotation["doc_id"], "log": clog})

        summaries_dict_scst = {**summaries_dict, "LED": scst_summary}
        kg_refined_scst, _  = kg_iterative_refinement(summaries_dict_scst, test_annotation)
        lcs_fused_scst, _   = lcs_guided_sentence_fusion(kg_refined_scst, test_annotation, normalized_weights)

        full_pipeline, _ = copy_mechanism_post_process(
            lcs_fused_scst, test_annotation, span_index, source_sents_orig)
        ensemble_summaries["scst_led_copy"].append(lcs_fused_scst)
        ensemble_summaries["full_pipeline"].append(full_pipeline)

    print("✅ All summaries generated!")

    with open(os.path.join(OUTPUT_DIR, "copy_mechanism_logs.json"), 'w', encoding='utf8') as f:
        json.dump(copy_logs, f, indent=2, ensure_ascii=False)
    with open(os.path.join(OUTPUT_DIR, "lcs_fusion_logs.json"), 'w', encoding='utf8') as f:
        json.dump(fusion_logs, f, indent=2, ensure_ascii=False)

    # ── Evaluate ───────────────────────────────────────────────────────────
    print("\n" + "="*70)
    print("📊 EVALUATING ALL APPROACHES")
    print("="*70)

    results = {}
    approach_labels = {
        "BART":           "BART (base)",
        "LED":            "LED (base)",
        "PEGASUS":        "PEGASUS (base)",
        "LED_SCST":       "LED_SCST ➋ (balanced reward)",
        "voting":         "Ensemble-Voting",
        "weighted":       "Ensemble-Weighted",
        "ranking":        "Ensemble-Ranking",
        "kg_refined":     "KG-Diff (Novel 5+7)",
        "lcs_fused":      "LCS-Fusion (Novel 8)",
        "copy_mech":      "LCS + Copy-Mech ➊",
        "scst_led_copy":  "SCST-LED + LCS ➋",
        "full_pipeline":  "FULL: SCST+LCS+Copy ➊➋ ★"
    }

    for key in ["BART","LED","PEGASUS","LED_SCST"]:
        label = approach_labels[key]
        print(f"\n  Evaluating {label}...")
        metrics = compute_metrics(all_model_summaries[key], references)
        results[key] = metrics
        tag = " 🎯" if metrics["rougeL"] >= 0.34 else ""
        print(f"  ✅ R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
              f"RL:{metrics['rougeL']:.4f}{tag}  BS:{metrics['bertscore_f1']:.4f}")

    for key in ["voting","weighted","ranking","kg_refined","lcs_fused",
                "copy_mech","scst_led_copy","full_pipeline"]:
        label = approach_labels[key]
        print(f"\n  Evaluating {label}...")
        metrics = compute_metrics(ensemble_summaries[key], references)
        results[key] = metrics
        tag = " 🎯" if metrics["rougeL"] >= 0.34 else ""
        print(f"  ✅ R1:{metrics['rouge1']:.4f}  R2:{metrics['rouge2']:.4f}  "
              f"RL:{metrics['rougeL']:.4f}{tag}  BS:{metrics['bertscore_f1']:.4f}")

    # ── Results Table ──────────────────────────────────────────────────────
    print("\n" + "="*85)
    print("📊 FINAL RESULTS SUMMARY")
    print("="*85)
    print(f"\n{'Approach':<45} {'R1':<8} {'R2':<8} {'RL':<12} {'BS-F1'}")
    print("-"*85)

    for key, label in approach_labels.items():
        if key not in results: continue
        m   = results[key]
        rl  = m["rougeL"]
        tag = "✓ ≥0.34 🎯" if rl >= 0.34 else f"({0.34-rl:.4f} short)"
        star = " ★" if key == "full_pipeline" else ""
        print(f"{label:<45} {m['rouge1']:<8.4f} {m['rouge2']:<8.4f} "
              f"{rl:.4f} {tag:<14} {m['bertscore_f1']:.4f}{star}")

    # ── Reward hacking comparison ──────────────────────────────────────────
    print("\n" + "="*85)
    print("🔬 REWARD HACKING ANALYSIS — Original vs Fixed SCST")
    print("="*85)
    print(f"  {'Metric':<22} {'Original (R1-only)':<22} {'Fixed (R1+R2+RL)':<22} {'Change'}")
    print(f"  {'-'*80}")

    if "LED_SCST" in results and "LED" in results:
        original_reported = {"rouge1": 0.5087, "rouge2": 0.2585, "rougeL": 0.2456}
        fixed = results["LED_SCST"]
        for metric in ["rouge1", "rouge2", "rougeL"]:
            orig_val  = original_reported[metric]
            fixed_val = fixed[metric]
            delta     = fixed_val - orig_val
            sign      = "+" if delta >= 0 else ""
            flag      = " ✅ improved" if (metric in ["rouge2","rougeL"] and delta > 0) else \
                        " ⚠️  reduced"  if (metric == "rouge1" and delta < 0) else ""
            print(f"  {metric.upper():<22} {orig_val:<22.4f} {fixed_val:<22.4f} {sign}{delta:.4f}{flag}")

    best_rl = max(results.items(), key=lambda x: x[1]["rougeL"])
    best_r2 = max(results.items(), key=lambda x: x[1]["rouge2"])
    print("\n" + "="*85)
    print(f"🏆 BEST ROUGE-L: {approach_labels.get(best_rl[0], best_rl[0])} → {best_rl[1]['rougeL']:.4f}")
    print(f"🏆 BEST ROUGE-2: {approach_labels.get(best_r2[0], best_r2[0])} → {best_r2[1]['rouge2']:.4f}")

    # ── Save summaries + results ───────────────────────────────────────────
    print("\n💾 Saving summaries and results...")
    for key in ["full_pipeline","copy_mech","scst_led_copy","lcs_fused"]:
        out_path = os.path.join(OUTPUT_DIR, f"{key}_summaries.json")
        source   = all_model_summaries.get(key) or ensemble_summaries.get(key, [])
        with open(out_path, 'w', encoding='utf8') as f:
            json.dump([{"id": item.get("id", i), "generated": summ, "reference": ref}
                       for i, (item, summ, ref) in enumerate(zip(test_data, source, references))],
                      f, indent=2, ensure_ascii=False)

    final_output = {
        "experiment":    "ROUGE-L > 0.34: Balanced SCST + Copy Mechanism + LCS Fusion",
        "reward_hacking_fix": {
            "problem":  "Original SCST used ROUGE-L-only reward → model repeated unigrams → R1=0.51 (inflated), RL=0.24 (worst)",
            "solution": f"Balanced reward: {REWARD_W_R1}·R1 + {REWARD_W_R2}·R2 + {REWARD_W_RL}·RL",
            "why_r2_prevents_hacking": "Repeating unigrams hurts bigram precision: repeated bigrams unlikely match reference bigrams",
            "why_rl_prevents_hacking": "Repeating tokens inflates length, reducing LCS/len precision ratio"
        },
        "scst_config": {
            "reward_weights":  {"r1": REWARD_W_R1, "r2": REWARD_W_R2, "rl": REWARD_W_RL},
            "rl_weight":       SCST_RL_WEIGHT,
            "lr":              SCST_LR,
            "epochs":          SCST_EPOCHS,
            "sample_temp":     SCST_SAMPLE_TEMP,
            "hacking_monitor": {
                "rep_rate_threshold":   HACK_REPETITION_THRESHOLD,
                "sent_len_range":       [HACK_MIN_SENT_LEN, HACK_MAX_SENT_LEN],
                "r1_rl_ratio_max":      HACK_R1_RL_RATIO_MAX,
                "vocab_diversity_min":  HACK_VOCAB_DIVERSITY_MIN,
                "signals_to_stop":      HACK_SIGNALS_TO_STOP
            }
        },
        "results":     results,
        "best_rougeL": {"name": best_rl[0], "metrics": best_rl[1]},
        "best_rouge2": {"name": best_r2[0], "metrics": best_r2[1]}
    }

    results_path = os.path.join(OUTPUT_DIR, "complete_results_balanced_scst.json")
    with open(results_path, 'w', encoding='utf8') as f:
        json.dump(final_output, f, indent=2, ensure_ascii=False)
    print(f"✅ Complete results saved to: {results_path}")

    print("\n" + "="*70)
    print("✅ PIPELINE COMPLETE!")
    print(f"   ➊ Copy Mechanism           → verbatim span extraction")
    print(f"   ➋ Balanced SCST (FIXED)    → {REWARD_W_R1}·R1 + {REWARD_W_R2}·R2 + {REWARD_W_RL}·RL reward")
    print(f"      No more unigram hacking  → R2+RL should improve vs original")
    print(f"   ★ Full Pipeline             → ROUGE-L ≥ 0.34 target")
    print("="*70)


if __name__ == "__main__":
    try:
        main()
    except KeyboardInterrupt:
        print("\n\n⚠️  Pipeline interrupted by user")
    except Exception as e:
        import traceback
        print(f"\n\n❌ Error: {e}")
        traceback.print_exc()

📥 Downloading NLTK data...
🚀 Device: cuda

📚 Loading HSLN role classifier...
✅ HSLN loaded

📚 Loading InLegalBERT...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


✅ InLegalBERT loaded

📚 Loading summarization models...
  Loading BART...
  ✅ BART loaded
  Loading LED...
  ✅ LED loaded
  Loading PEGASUS...
  ✅ PEGASUS loaded
✅ All models loaded

📊 Loading evaluation metrics...

🚀 ENSEMBLE SUMMARIZATION — ROUGE-L > 0.34
   + ➊ Copy-Mechanism Verbatim Span Extraction
   + ➋ SCST RL (FIXED: balanced R1+R2+RL reward)
      FIX: replaces ROUGE-L-only reward that caused hacking
           (was: R1=0.51 inflated, RL=0.24 worst)
           (fix: 0.2·R1 + 0.4·R2 + 0.4·RL)
   + Novel Idea 8: LCS-Guided Sentence Fusion (existing)

📂 Loading data...
   Full train set : 7030 documents  (used for role contribution)
   SCST train set : 3000 documents  (capped by MAX_TRAIN_SAMPLES)
   Test set       : 100 documents

📝 Annotating FULL training set (7030 docs)...

📝 Annotating 7030 documents with 8-label roles...


100%|███████████████████████████████████████████████████████████████████████████████| 7030/7030 [49:44<00:00,  2.36it/s]


✅ Annotations saved: ./ensemble_results_rougeL_boost/sentence_role_annotations_8label.json (7030 docs)

📝 Annotating SCST training subset (3000 docs)...

📝 Annotating 3000 documents with 8-label roles...


...

    [CopyMech] Building suffix index over 59 source sentences...
    [CopyMech] Index built: 27,919 unique n-grams, 28,300 total entries


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy/util.py:969: UserWarning: [W095] Model 'en_legal_ner_trf' (3.2.0) was trained with spaCy v3.2.2 and may not be 100% compatible with the current version (3.8.11). If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current spaCy version. For more details and available updates, run: python -m spacy validate
  warnings.warn(warn_msg)
/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/spacy_transformers/layers/hf_shim.py:137: UserWarning: Error loading saved torch state_dict with strict=True, likely due to differences between 'transformers' versions. Attempting to load with strict=False as a fallback...

If you see errors or degraded performance, download a newer compatible model or retrain your custom model with the current 'transformers' and 'spacy-transformers' versions. For more details and available updates, run: python -m spacy validate
  war

✅ All summaries generated!

📊 EVALUATING ALL APPROACHES

  Evaluating BART (base)...


/home/RSlab/.conda/envs/legal-nlp/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.weight', 'roberta.pooler.dense.bias']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  ✅ R1:0.3743  R2:0.1939  RL:0.2134  BS:0.8502

  Evaluating LED (base)...
  ✅ R1:0.3924  R2:0.2224  RL:0.2437  BS:0.8491

  Evaluating PEGASUS (base)...
  ✅ R1:0.3815  R2:0.1904  RL:0.2123  BS:0.8425

  Evaluating LED_SCST ➋ (balanced reward)...
  ✅ R1:0.3924  R2:0.2224  RL:0.2437  BS:0.8491

  Evaluating Ensemble-Voting...
  ✅ R1:0.4610  R2:0.2447  RL:0.2398  BS:0.8469

  Evaluating Ensemble-Weighted...
  ✅ R1:0.3459  R2:0.1861  RL:0.2081  BS:0.8469

  Evaluating Ensemble-Ranking...
  ✅ R1:0.5142  R2:0.2697  RL:0.2532  BS:0.8421

  Evaluating KG-Diff (Novel 5+7)...
  ✅ R1:0.3924  R2:0.2224  RL:0.2437  BS:0.8491

  Evaluating LCS-Fusion (Novel 8)...
  ✅ R1:0.3940  R2:0.2223  RL:0.2408  BS:0.8490

  Evaluating LCS + Copy-Mech ➊...
  ✅ R1:0.3940  R2:0.2216  RL:0.2406  BS:0.8443

  Evaluating SCST-LED + LCS ➋...
  ✅ R1:0.3940  R2:0.2223  RL:0.2408  BS:0.8490

  Evaluating FULL: SCST+LCS+Copy ➊➋ ★...
  ✅ R1:0.3940  R2:0.2216  RL:0.2406  BS:0.8443

📊 FINAL RESULTS SUMMARY

Approach        